# Bakaano-Hydro Beginner Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/confidence-duku/bakaano-hydro/blob/main/Bakaano-Hydro%20Beginner%20Colab.ipynb)

This is the official Colab notebook for the core beginner workflow.


## Official Path

This notebook follows the recommended public workflow for most users:

1. End-to-end workflow: preprocessing -> runoff/routing -> training.
2. Evaluation workflow: run `evaluate_streamflow_model_interactively(...)` with the trained model.
3. Simulation workflow: prefer `simulate_grdc_csv_stations(...)` for official station sets; use `simulate_streamflow(...)` only for custom lat/lon points.

Use `ProjectContext` for shared paths/readiness checks. Advanced modules such as `FloodMapper` and `ScenarioManager` are extensions that usually come after the core training/evaluation/simulation workflow is working.


## Set Colab Runtime to GPU

Use **Runtime -> Change runtime type -> GPU** before training.


In [ ]:
!pip -q uninstall -y torch torchvision torchaudio
!pip -q install "bakaano-hydro[gpu] @ git+https://github.com/confidence-duku/bakaano-hydro.git"
!pip -q install h5netcdf


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    raise RuntimeError('No GPU detected. In Colab, set Runtime -> Change runtime type -> GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# User Inputs (edit this cell only)
# ----------------------------------------------------------------------------
# 1) Workspace and basin shapefile
folder_name = 'bakaano_workflow'
shapefile_name = 'your_basin.shp'  # must exist in MyDrive/<folder_name>/shapes/

# 2) Observed-data mode (choose one): 'GRDC' or 'CSV'
OBSERVED_DATA_MODE = 'GRDC'

# GRDC mode input (used when OBSERVED_DATA_MODE='GRDC')
grdc_filename = 'your_grdc_data.nc'  # file in MyDrive/<folder_name>/data/

# CSV mode inputs (used when OBSERVED_DATA_MODE='CSV')
csv_dir_name = 'csv_timeseries'      # folder in MyDrive/<folder_name>/data/
lookup_csv_name = 'station_lookup.csv'  # file in MyDrive/<folder_name>/data/

# 3) Global model settings
climate_data_source = 'ERA5'  # must be one of: ERA5, CHIRPS, CHELSA
routing_method = 'mfd'        # mfd, d8, dinf

# 4) Run toggles
RUN_PREPROCESS = True
RUN_RUNOFF_ROUTING = True
RUN_TRAIN = True
RUN_EVAL = True
RUN_SIM_GRDC = True
RUN_SIM_POINTS = True
RUN_FLOOD_MAPPING = False

# 5) Dates (single shared window for beginner workflow)
# These are reused by preprocessing, runoff routing, interactive exploration, and AlphaEarth.
WORKFLOW_START_DATE = '2001-01-01'
WORKFLOW_END_DATE = '2010-12-31'
FLOOD_RETURN_PERIOD = 10

# Derived dates (kept explicit so downstream cells stay readable)
TRAIN_START_DATE = '2001-01-01'
TRAIN_END_DATE = '2020-12-31'

EVAL_START_DATE = '2001-01-01'
EVAL_END_DATE = '2020-12-31'




In [ ]:
# Optional Advanced Settings (optional; beginners can skip editing this cell)
batch_size = 32
num_epochs = 300
learning_rate = 1e-3
loss_function = 'mse'
area_normalize = True  # keep this consistent across training, evaluation, and simulation
log_transform = True  # set False only when training and inference should stay in linear space
# Training saves area_normalize and log_transform with the model; inference loads them automatically.
model_overwrite = True

LR_SCHEDULE = 'cosine'
WARMUP_EPOCHS = 5
MIN_LEARNING_RATE = 1e-5

CSV_ID_COL = 'id'
CSV_LAT_COL = 'latitude'
CSV_LON_COL = 'longitude'
CSV_DATE_COL = 'date'
CSV_DISCHARGE_COL = 'discharge'
CSV_FILE_PATTERN = '{id}.csv'


In [ ]:
from pathlib import Path
import shutil

working_dir_drive = Path('/content/drive/MyDrive') / folder_name
working_dir_local = Path('/content') / folder_name
working_dir = working_dir_drive
study_area = working_dir_drive / 'shapes' / shapefile_name

if OBSERVED_DATA_MODE == 'GRDC':
    grdc_netcdf = working_dir_drive / 'data' / grdc_filename
    csv_dir = None
    lookup_csv = None
elif OBSERVED_DATA_MODE == 'CSV':
    grdc_netcdf = None
    csv_dir = working_dir_drive / 'data' / csv_dir_name
    lookup_csv = working_dir_drive / 'data' / lookup_csv_name
else:
    raise ValueError("OBSERVED_DATA_MODE must be 'GRDC' or 'CSV'.")

working_dir_drive.mkdir(parents=True, exist_ok=True)
(working_dir_drive / 'shapes').mkdir(parents=True, exist_ok=True)
(working_dir_drive / 'data').mkdir(parents=True, exist_ok=True)
working_dir_local.mkdir(parents=True, exist_ok=True)
(working_dir_local / 'shapes').mkdir(parents=True, exist_ok=True)
(working_dir_local / 'data').mkdir(parents=True, exist_ok=True)

print('working_dir (drive):', working_dir_drive)
print('working_dir (local):', working_dir_local)
print('study_area:', study_area)
print('mode:', OBSERVED_DATA_MODE)
print('grdc_netcdf:', grdc_netcdf)
print('csv_dir:', csv_dir)
print('lookup_csv:', lookup_csv)


## Configuration

1. Fill **User Inputs**.
2. Optionally edit **Advanced Settings**.
3. Run cells top-to-bottom.


In [ ]:
if not study_area.exists():
    raise FileNotFoundError(f'study_area not found: {study_area}')

grdc_mode = grdc_netcdf is not None
csv_mode = (csv_dir is not None) and (lookup_csv is not None)

if grdc_mode == csv_mode:
    raise ValueError(
        'Choose exactly one observed-data mode:\n'
        "  1) GRDC mode: set grdc_netcdf=Path(...), csv_dir=None, lookup_csv=None\n"
        "  2) CSV mode: set grdc_netcdf=None, csv_dir=Path(...), lookup_csv=Path(...)"
    )

grdc_netcdf_runtime = None

def open_grdc_with_fallback(nc_path):
    import xarray as xr
    errors = []
    for engine in [None, 'h5netcdf']:
        try:
            if engine is None:
                return xr.open_dataset(nc_path), 'netcdf4(default)'
            return xr.open_dataset(nc_path, engine=engine), engine
        except Exception as e:
            name = 'netcdf4(default)' if engine is None else engine
            errors.append(f'{name}: {e}')
    raise RuntimeError('Unable to open GRDC NetCDF with available backends:\n' + '\n'.join(errors))

if grdc_mode:
    if not grdc_netcdf.exists():
        raise FileNotFoundError(f'grdc_netcdf not found: {grdc_netcdf}')

    src_size = grdc_netcdf.stat().st_size
    if src_size <= 0:
        raise RuntimeError(f'GRDC file is empty: {grdc_netcdf}')

    usage = shutil.disk_usage('/content')
    if usage.free < src_size + 200 * 1024 * 1024:
        raise RuntimeError(
            'Not enough free space in /content to stage GRDC NetCDF. '
            f'Need at least {src_size + 200 * 1024 * 1024} bytes.'
        )

    grdc_netcdf_runtime = working_dir_local / 'grdc' / grdc_netcdf.name
    grdc_netcdf_runtime.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(grdc_netcdf, grdc_netcdf_runtime)

    dst_size = grdc_netcdf_runtime.stat().st_size
    if dst_size != src_size:
        raise RuntimeError(
            'GRDC file copy size mismatch; copy may be incomplete. '
            f'source={src_size} bytes, copied={dst_size} bytes'
        )

    try:
        ds_chk, engine_used = open_grdc_with_fallback(grdc_netcdf_runtime)
        with ds_chk:
            _ = tuple(ds_chk.dims.keys())
        print('GRDC runtime copy ready:', grdc_netcdf_runtime)
        print('GRDC backend:', engine_used)
    except Exception as e:
        raise RuntimeError(
            f'Failed to open GRDC NetCDF after local copy: {grdc_netcdf_runtime}\n'
            'If this file opens on HPC but not on Colab, install h5netcdf and retry:\n'
            '  !pip -q install h5netcdf\n'
            f'Details: {e}'
        ) from e

if csv_mode:
    if not csv_dir.exists():
        raise FileNotFoundError(f'csv_dir not found: {csv_dir}')
    if not lookup_csv.exists():
        raise FileNotFoundError(f'lookup_csv not found: {lookup_csv}')

print('Input validation passed.')
print('Mode:', 'GRDC' if grdc_mode else 'CSV')


## 1) Download and preprocess input data

Run this section to prepare meteo, NDVI, soil, DEM, and catchment inputs.

This is the hydrologic foundation of the workflow. These layers describe basin structure, climate forcing, and land-surface conditions that control how water is stored, partitioned, and routed downstream.


In [ ]:
import ee
ee.Authenticate(auth_mode="notebook")   # forces link + paste code flow
ee.Initialize()

In [ ]:
if RUN_PREPROCESS:
    from bakaano.data.dem import DEM
    from bakaano.data.tree_cover import TreeCover
    from bakaano.data.ndvi import NDVI
    from bakaano.data.soil import Soil
    from bakaano.data.alpha_earth import AlphaEarth
    from bakaano.data.meteo import Meteo

    dd = DEM(
        working_dir=str(working_dir),
        study_area=str(study_area),
        local_data=False,
        local_data_path=None,
    )
    dd.get_dem_data()

    vf = TreeCover(
        working_dir=str(working_dir),
        study_area=str(study_area),
        start_date=WORKFLOW_START_DATE,
        end_date=WORKFLOW_END_DATE,
    )
    vf.get_tree_cover_data()

    nd = NDVI(
        working_dir=str(working_dir),
        study_area=str(study_area),
        start_date=WORKFLOW_START_DATE,
        end_date=WORKFLOW_END_DATE,
    )
    nd.get_ndvi_data()

    sgd = Soil(
        working_dir=str(working_dir),
        study_area=str(study_area),
    )
    sgd.get_soil_data()

    ae = AlphaEarth(
        working_dir=str(working_dir),
        study_area=str(study_area),
        start_date=WORKFLOW_START_DATE,
        end_date=WORKFLOW_END_DATE,
    )
    ae.get_alpha_earth()

    cd = Meteo(
        working_dir=str(working_dir_local),
        study_area=str(study_area),
        start_date=WORKFLOW_START_DATE,
        end_date=WORKFLOW_END_DATE,
        local_data=False,
        data_source=climate_data_source,
    )
    cd.get_meteo_data()
    src_climate = working_dir_local / climate_data_source
    dst_climate = working_dir_drive / climate_data_source
    if src_climate.exists():
        shutil.copytree(src_climate, dst_climate, dirs_exist_ok=True)
        print(f'Synced climate outputs to Drive: {dst_climate}')

    print('Preprocessing complete.')
else:
    print('Skipping preprocessing step.')


### Optional quick plots

Use plotting helpers to quickly inspect inputs if needed.

A quick visual check helps confirm that the hydrologic inputs look physically reasonable before they are used to generate runoff and support model training.


## 2) Compute runoff and route to river network

Runs runoff generation and routing using prepared inputs.

This step converts gridded forcing and landscape information into routed runoff signals. It is the main bridge from physical hydrology to machine learning, because the model later learns from these flow-aware predictors rather than from raw inputs alone.

If `runoff_output/rainfall_sparse_arrays.pkl` is also available for the same date range, Bakaano will automatically use routed rainfall as an extra temporal predictor during training and simulation.


In [ ]:
if RUN_RUNOFF_ROUTING:
    from bakaano.hydrology.veget import VegET

    # Stage required inputs from Drive to local runtime.
    for folder in ['shapes', 'elevation', 'soil', 'ndvi', 'vcf', 'alpha_earth', climate_data_source]:
        src = working_dir_drive / folder
        dst = working_dir_local / folder
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)

    # Stage prior runoff outputs/checkpoints so compute can be skipped on fresh runtimes.
    for folder in ['runoff_output', 'catchment']:
        src = working_dir_drive / folder
        dst = working_dir_local / folder
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)

    local_study_area = working_dir_local / 'shapes' / shapefile_name
    local_final_file = working_dir_local / 'runoff_output' / 'wacc_sparse_arrays.pkl'

    if local_final_file.exists():
        print(f'Found existing routed runoff at {local_final_file}. Skipping VegET compute.')
    else:
        vg = VegET(
            working_dir=str(working_dir_local),
            study_area=str(local_study_area),
            start_date=WORKFLOW_START_DATE,
            end_date=WORKFLOW_END_DATE,
            climate_data_source=climate_data_source,
            routing_method=routing_method,
        )
        vg.compute_veget_runoff_route_flow(
            resume=True,
            checkpoint_days=30,
        )

    # Sync latest local outputs back to Drive.
    for folder in ['runoff_output', 'catchment']:
        src = working_dir_local / folder
        dst = working_dir_drive / folder
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print('Runoff and routing complete.')
else:
    print('Skipping runoff/routing step.')


In [ ]:
from bakaano.hydrology.plot_runoff import RoutedRunoff

rr = RoutedRunoff(
    working_dir=str(working_dir),
    study_area=str(study_area),
)

rr.map_routed_runoff(date='2003-07-07', vmax=7)


## 3) Interactive exploration

Use this section for quick map and station-level checks.

These checks help verify that the routed hydrologic signals align with the basin and station locations before training. They provide a quick sanity check on the predictors the ML model will actually see.


In [ ]:
from bakaano.core.project import ProjectContext

project = ProjectContext(
    working_dir=str(working_dir),
    study_area=str(study_area),
    climate_data_source=climate_data_source,
)

print('Interactive exploration has been removed from Bakaano-Hydro.')


### Routed runoff timeseries

Select a station ID to inspect routed runoff timeseries.

At this stage you are inspecting the time-varying hydrologic input that underpins streamflow prediction. If timing or magnitude looks wrong here, the ML model will struggle regardless of its architecture.


In [ ]:
if grdc_netcdf_runtime is not None and grdc_netcdf_runtime.exists():
    rr.interactive_plot_routed_runoff_timeseries(
        start_date=WORKFLOW_START_DATE,
        end_date=WORKFLOW_END_DATE,
        grdc_netcdf=str(grdc_netcdf_runtime),
    )
elif lookup_csv is not None and lookup_csv.exists():
    rr.interactive_plot_routed_runoff_timeseries(
        start_date=WORKFLOW_START_DATE,
        end_date=WORKFLOW_END_DATE,
        lookup_csv=str(lookup_csv),
        id_col=CSV_ID_COL,
        lat_col=CSV_LAT_COL,
        lon_col=CSV_LON_COL,
    )
else:
    print('Set grdc_netcdf or lookup_csv before running interactive routed runoff timeseries.')


## 4) Train model

Train with GRDC NetCDF or station CSV inputs, then save the model.

Here the neural network learns the mapping from routed hydrologic predictors and static catchment descriptors to observed discharge at gauges. The hydrology provides physical structure and context, while the ML model learns the flexible response needed for prediction.


In [ ]:
from bakaano.neuralnet.train import train_streamflow_model

if RUN_TRAIN:
    if grdc_netcdf_runtime is not None and grdc_netcdf_runtime.exists():
        train_streamflow_model(
            working_dir=str(working_dir),
            study_area=str(study_area),
            train_start=TRAIN_START_DATE,
            train_end=TRAIN_END_DATE,
            grdc_netcdf=str(grdc_netcdf_runtime),
            batch_size=batch_size,
            num_epochs=num_epochs,
            learning_rate=learning_rate,
            loss_function=loss_function,
            lr_schedule=LR_SCHEDULE,
            warmup_epochs=WARMUP_EPOCHS,
            min_learning_rate=MIN_LEARNING_RATE,
            routing_method=routing_method,
            area_normalize=area_normalize,
            log_transform=log_transform,
            model_overwrite=model_overwrite,
        )
    else:
        train_streamflow_model(
            working_dir=str(working_dir),
            study_area=str(study_area),
            train_start=TRAIN_START_DATE,
            train_end=TRAIN_END_DATE,
            grdc_netcdf=None,
            batch_size=batch_size,
            num_epochs=num_epochs,
            learning_rate=learning_rate,
            loss_function=loss_function,
            lr_schedule=LR_SCHEDULE,
            warmup_epochs=WARMUP_EPOCHS,
            min_learning_rate=MIN_LEARNING_RATE,
            routing_method=routing_method,
            area_normalize=area_normalize,
            log_transform=log_transform,
            csv_dir=str(csv_dir),
            lookup_csv=str(lookup_csv),
            id_col=CSV_ID_COL,
            lat_col=CSV_LAT_COL,
            lon_col=CSV_LON_COL,
            date_col=CSV_DATE_COL,
            discharge_col=CSV_DISCHARGE_COL,
            file_pattern=CSV_FILE_PATTERN,
            model_overwrite=model_overwrite,
        )
else:
    print('Skipping training.')


## 5) Evaluate model

Loads a trained model and compares predictions with observations.

Evaluation tests whether the learned ML relationship reproduces hydrologic behavior seen in observations, including timing, recession patterns, and peak-flow response.


In [ ]:
from bakaano.neuralnet.simulate import evaluate_streamflow_model_interactively

# Re-train models after changing configuration such as area_normalize or response scaling behavior.
model_path = working_dir / 'models' / 'bakaano_model.keras'
print('model_path exists:', model_path.exists())

if RUN_EVAL and model_path.exists():
    if grdc_netcdf_runtime is not None and grdc_netcdf_runtime.exists():
        evaluate_streamflow_model_interactively(
            working_dir=str(working_dir),
            study_area=str(study_area),
            model_path=str(model_path),
            val_start=EVAL_START_DATE,
            val_end=EVAL_END_DATE,
            grdc_netcdf=str(grdc_netcdf_runtime),
            routing_method=routing_method,
        )
    else:
        evaluate_streamflow_model_interactively(
            working_dir=str(working_dir),
            study_area=str(study_area),
            model_path=str(model_path),
            val_start=EVAL_START_DATE,
            val_end=EVAL_END_DATE,
            grdc_netcdf=None,
            routing_method=routing_method,
            csv_dir=str(csv_dir),
            lookup_csv=str(lookup_csv),
            id_col=CSV_ID_COL,
            lat_col=CSV_LAT_COL,
            lon_col=CSV_LON_COL,
            date_col=CSV_DATE_COL,
            discharge_col=CSV_DISCHARGE_COL,
            file_pattern=CSV_FILE_PATTERN,
        )
else:
    print('Skipping evaluation.')


## 6) Simulate streamflow

Run simulations for stations or custom lat/lon points.

Once trained, the model can transfer what it learned from gauged basins to station sets or user-defined points. The routed hydrologic inputs still carry the basin dynamics, and the ML model produces the final discharge estimate.


In [ ]:
from bakaano.neuralnet.simulate import simulate_grdc_csv_stations

if model_path.exists() and RUN_SIM_GRDC:
    if grdc_netcdf_runtime is not None and grdc_netcdf_runtime.exists():
        simulate_grdc_csv_stations(
            working_dir=str(working_dir),
            study_area=str(study_area),
            model_path=str(model_path),
            sim_start=WORKFLOW_START_DATE,
            sim_end=WORKFLOW_END_DATE,
            grdc_netcdf=str(grdc_netcdf_runtime),
            routing_method=routing_method,
        )
    else:
        simulate_grdc_csv_stations(
            working_dir=str(working_dir),
            study_area=str(study_area),
            model_path=str(model_path),
            sim_start=WORKFLOW_START_DATE,
            sim_end=WORKFLOW_END_DATE,
            grdc_netcdf=None,
            routing_method=routing_method,
            csv_dir=str(csv_dir),
            lookup_csv=str(lookup_csv),
            id_col=CSV_ID_COL,
            lat_col=CSV_LAT_COL,
            lon_col=CSV_LON_COL,
            date_col=CSV_DATE_COL,
            discharge_col=CSV_DISCHARGE_COL,
            file_pattern=CSV_FILE_PATTERN,
        )
else:
    print('Skipping station simulation.')


In [ ]:
from bakaano.neuralnet.simulate import simulate_streamflow

SIM_POINT_LATLIST = [13.8, 13.9] #user provided latitudes for point simulation
SIM_POINT_LONLIST = [3.0, 4.0] #user provided longitudes for point simulation

if model_path.exists() and RUN_SIM_POINTS:
    simulate_streamflow(
        working_dir=str(working_dir),
        study_area=str(study_area),
        model_path=str(model_path),
        sim_start=WORKFLOW_START_DATE,
        sim_end=WORKFLOW_END_DATE,
        latlist=SIM_POINT_LATLIST,
        lonlist=SIM_POINT_LONLIST,
        routing_method=routing_method,
    )
else:
    print('Skipping point simulation.')


In [ ]:
import glob
import pandas as pd

pred_files = sorted(glob.glob(str(working_dir / 'predicted_streamflow_data' / '*.csv')))
print('Prediction files:', len(pred_files))
if pred_files:
    print('Example:', pred_files[0])
    df = pd.read_csv(pred_files[0])
    display(df.head())